# California Language Archive

Scrapes resource metadata and sizes from the California Language Archive.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from shapely.geometry import Point
import geopandas
import matplotlib.pyplot as plt
import json
import re

In [ ]:
cla = 'https://cla.berkeley.edu/browse/language-list.html'

In [ ]:
html_content = requests.get(cla).content

soup = BeautifulSoup(html_content, 'html.parser')

In [ ]:
language_links = soup.select('div.group a')

In [ ]:
language_names = [link.text.strip() for link in language_links]

In [ ]:
language_names = [name for name in language_names if name.strip()]

In [ ]:
language_names

In [ ]:
len(language_names)

In [ ]:
glottolog_data = pd.read_csv('languages_and_dialects_geo.csv')

In [ ]:
glottolog_data

In [ ]:
df_filtered = glottolog_data[glottolog_data['name'].isin(language_names)]

In [ ]:
df_filtered

In [ ]:
points = df_filtered.apply(lambda r: Point(r.longitude, r.latitude), axis=1)
gdb = geopandas.GeoDataFrame(df_filtered.assign(geometry=points), \
                        crs={'init' :'epsg:4326'})
xys = gdb['geometry'].apply(lambda pt: pd.Series({'x': pt.x, 'y': pt.y}))
gdb['x'] = xys['x']
gdb['y'] = xys['y']

In [ ]:
worldmap = geopandas.read_file(geopandas.datasets.get_path("naturalearth_lowres"))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
worldmap.plot(color="white", edgecolor = 'black',ax=ax, linewidth=0.3)
gdb.plot(ax=ax, color = 'blue', markersize = 5)
plt.show()

In [ ]:
items = 'https://cla.berkeley.edu/list/index.html?size=25&collfrom=0&bndlfrom=0&collpage=0&bndlpage=0&tab=bndl&with_js=1'

In [ ]:
url = 'https://xfhpwly456.execute-api.us-east-1.amazonaws.com/api/sq'
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:124.0) Gecko/20100101 Firefox/124.0',
    'Accept': 'application/json, text/javascript, */*; q=0.01',
    'Accept-Language': 'en-GB,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'Origin': 'https://cla.berkeley.edu',
    'DNT': '1',
    'Sec-GPC': '1',
    'Connection': 'keep-alive',
    'Referer': 'https://cla.berkeley.edu/',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'cross-site',
    'TE': 'trailers'
}

data = {
    "qstr": [],
    "tab": "bndl",
    "with_js": "1",
    "size": "25",
    "bndlfrom": "0"
}

# Iterate over different values of bndlfrom
for i in range(1, 5001, 500):
    data['bndlfrom'] = str(i)
    response = requests.post(url, headers=headers, json=data)
    print(f"Request for bndlfrom={i} - Status code: {response.status_code}")

In [ ]:
response.content

In [ ]:
len(response.content)

In [ ]:
len(data)

In [ ]:
f = open('./CLA/cla.json')

In [ ]:
data = json.load(f)

In [ ]:
data

In [ ]:
bndl_data = data['bndl']

In [ ]:
bndl_data

In [ ]:
data = []
for item in bndl_data['hits']['hits']:
    sort = item['sort'][0]
    ul_md = item['_source']['ul_md']
    
    # Applying regex to extract language name
    language_match = re.search(r'<span class=\'label\'>Language:</span>&nbsp;<a .*?>(.*?)</a>', ul_md)
    if language_match:
        language = language_match.group(1)
    else:
        language = None
    
    data.append({'name': sort, 'language': language, 'ul_md': ul_md})

df = pd.DataFrame(data)
print(df)

In [ ]:
k = df['ul_md'].to_list()

In [ ]:
k

In [ ]:
def extract_file_info(ul_md):
    soup = BeautifulSoup(ul_md, 'html.parser')
    detailonly_element = soup.find('p', class_='detailonly')
    if detailonly_element:
        file_info_list = []
        size_info_list = []
        br_tags = detailonly_element.find_all('br')
        if br_tags:
            for br_tag in br_tags:
                next_sibling = br_tag.next_sibling
                if next_sibling is not None:
                    br_text = br_tag.next_sibling.strip()
                    match = re.match(r'(.+)\s+\((\d+)\s+bytes\)', br_text)
                    if match:
                        file_info = match.group(1)
                        size_info = match.group(2)
                        file_info_list.append(file_info)
                        size_info_list.append(size_info) 
        if file_info_list and size_info_list:
            return file_info_list, size_info_list
    return 'not found', 'not found'

In [ ]:
tr = [extract_file_info(i) for i in k]

In [ ]:
tr

In [ ]:
data = []
for item in bndl_data['hits']['hits']:
    sort = item['sort'][0]
    ul_md = item['_source']['ul_md']
    
    # Extracting language name
    language_match = re.search(r'<span class=\'label\'>Language:</span>&nbsp;<a .*?>(.*?)</a>', ul_md)
    if language_match:
        language = language_match.group(1)
    else:
        language = None
    
    # Extracting file information
    file_info_list, size_info_list = extract_file_info(ul_md)
    print(file_info_list)
    
    # Creating data dictionary
    data_dict = {'name': sort, 'language': language, 'ul_md': ul_md}
    
    # If file information is available, add it to the dictionary
    if file_info_list and size_info_list:
        data_dict['file_info'] = file_info_list
        data_dict['file_size'] = size_info_list
    
    data.append(data_dict)

# Create DataFrame
df = pd.DataFrame(data)
print(df)

In [ ]:
df

In [ ]:
def process_json(json_file):
    data_2 = open(json_file)
    data_2 = json.load(data_2)
    bndl_data = data_2['bndl']
    data = []
    for item in bndl_data['hits']['hits']:
        sort = item['sort'][0]
        ul_md = item['_source']['ul_md']
        
        # Extracting language name
        language_match = re.search(r'<span class=\'label\'>Language:</span>&nbsp;<a .*?>(.*?)</a>', ul_md)
        if language_match:
            language = language_match.group(1)
        else:
            language = None
        
        # Extracting file information
        file_info_list, size_info_list = extract_file_info(ul_md)
        print(file_info_list)
        
        # Creating data dictionary
        data_dict = {'name': sort, 'language': language, 'ul_md': ul_md}
        
        # If file information is available, add it to the dictionary
        if file_info_list and size_info_list:
            data_dict['file_info'] = file_info_list
            data_dict['file_size'] = size_info_list
        
        data.append(data_dict)

    # Create DataFrame
    df = pd.DataFrame(data)
    return df

In [ ]:
data_2 = process_json('CLA/cla_2.json')

In [ ]:
data_2

In [ ]:
data_3 = process_json('CLA/cla_3.json')

In [ ]:
data_3

In [ ]:
data_4 = process_json('CLA/cla_3.json')

In [ ]:
data_4

In [ ]:
tr = [extract_file_info(i) for i in df['ul_md'].to_list()]

In [ ]:
tr

In [ ]:
data

In [ ]:
data = pd.DataFrame(data)

In [ ]:
data

In [ ]:
all_data = pd.concat([data, data_2, data_3, data_4])

In [ ]:
all_data

In [ ]:
def sum_integers(value):
    if value == 'not found':
        return 0
    elif isinstance(value, list):
        try:
            # Convert all items inside the list to integers
            value = [int(item) for item in value]
            return sum(value)  # Sum up the integers
        except ValueError:
            return 0  # Return 0 if any item inside the list is not convertible to an integer
    else:
        return value

In [ ]:
all_data['values'] = df['file_size'].apply(sum_integers)

In [ ]:
all_data

In [ ]:
grouped_df = all_data.groupby('language')['values'].sum().reset_index()
grouped_df['values'] = grouped_df['values'] / (1024 ** 3)
grouped_df.rename(columns={'values': 'value'}, inplace=True)

In [ ]:
grouped_df

In [ ]:
grouped_df = grouped_df.sort_values(by='value', ascending=False)

In [ ]:
grouped_df

In [ ]:
filtered_df = grouped_df[grouped_df['value'] > 0]

In [ ]:
filtered_df

In [ ]:
filtered_df.to_csv('CLA.csv', index=False)

In [ ]:
language_set = set(grouped_df['language'])

In [ ]:
items_not_in_df = [item for item in language_names if item not in language_set]

In [ ]:
len(items_not_in_df)

In [ ]:
items_not_in_df